# Dashboard Data Preparation
Genera los CSV de conteos que se usan en el dashboard (Stack Overflow Developer Survey).

Pestañas: **Uso actual**, **Tendencias futuras**, **Demografía**.

In [ ]:
import pandas as pd
import os

In [ ]:
!wget -O survey-data-updated.csv https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/HLOosvsPgIwt5dgOOh1RSg/survey-data-updated.csv

In [ ]:
df = pd.read_csv('survey-data-updated.csv')
print(df.shape)

# Verificar que existan las columnas que necesita el dashboard
needed = ['LanguageHaveWorkedWith','DatabaseHaveWorkedWith','PlatformHaveWorkedWith','WebframeHaveWorkedWith',
          'LanguageWantToWorkWith','DatabaseWantToWorkWith','PlatformWantToWorkWith','WebframeWantToWorkWith',
          'Country','Age','EdLevel']
print('Columnas faltantes:', [c for c in needed if c not in df.columns])

## Columnas con varios valores (separados por `;`)
Se separan con `split` + `explode` y se cuentan.

In [ ]:
os.makedirs('dashboard-data', exist_ok=True)

def multi_counts(col, top=None):
    s = df[col].dropna().str.split(';').explode().str.strip()
    out = s.value_counts().reset_index()
    out.columns = [col, 'Count']
    return out.head(top) if top else out

tech = {
    # Pestaña 1: Uso actual
    'LanguageHaveWorkedWith': 10,
    'DatabaseHaveWorkedWith': 10,
    'PlatformHaveWorkedWith': None,   # todas las plataformas
    'WebframeHaveWorkedWith': 10,
    # Pestaña 2: Tendencias futuras
    'LanguageWantToWorkWith': 10,
    'DatabaseWantToWorkWith': 10,
    'PlatformWantToWorkWith': None,
    'WebframeWantToWorkWith': 10,
}

for col, top in tech.items():
    out = multi_counts(col, top)
    out.to_csv(f'dashboard-data/{col}.csv', index=False)
    print(col, '->', len(out), 'filas')

## Pestaña 3: Demografía

In [ ]:
# Países (top 20)
country = df['Country'].value_counts().head(20).reset_index()
country.columns = ['Country', 'Count']
country.to_csv('dashboard-data/Country.csv', index=False)

# Grupos de edad (en orden lógico)
age_order = ['Under 18 years old', '18-24 years old', '25-34 years old',
             '35-44 years old', '45-54 years old', '55-64 years old',
             '65 years or older', 'Prefer not to say']
age = df['Age'].value_counts().reindex(age_order).dropna().reset_index()
age.columns = ['Age', 'Count']
age['Count'] = age['Count'].astype(int)
age.to_csv('dashboard-data/Age.csv', index=False)

# Nivel educativo
ed = df['EdLevel'].value_counts().reset_index()
ed.columns = ['EdLevel', 'Count']
ed.to_csv('dashboard-data/EdLevel.csv', index=False)

print(sorted(os.listdir('dashboard-data')))

## Vista previa rápida

In [ ]:
for f in sorted(os.listdir('dashboard-data')):
    print('\n' + f)
    print(pd.read_csv(f'dashboard-data/{f}').head(10).to_string(index=False))

## Empaquetar para descargar
Crea un solo `.zip` con los 11 CSV.

In [ ]:
import shutil
shutil.make_archive('dashboard-data', 'zip', 'dashboard-data')
'dashboard-data.zip'